1. Input Data

In [15]:
import numpy as np
import torch

data = np.load("frozen_soil_sensor_dataset_test_corrected.npz")

all_T_sensor_clean = data["all_T_sensor"]
params_arr = data["params_arr"]
curve_ids = data["curve_ids"]
t = data["t"]
sensor_z = data["sensor_z"]

# (n_case, Nt, n_sensor) -> (n_case, n_sensor, Nt)
X_sensor_clean = torch.from_numpy(
    all_T_sensor_clean
).permute(0, 2, 1).float()

print("all_T_sensor_clean:", all_T_sensor_clean.shape)
print("X_sensor_clean    :", X_sensor_clean.shape)
print("params_arr        :", params_arr.shape)

all_T_sensor_clean: (220, 61, 11)
X_sensor_clean    : torch.Size([220, 11, 61])
params_arr        : (220, 6)


2. 建立 Inverse DeepONet

2.1. 网络架构

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class BranchCNN(nn.Module):

    def __init__(self, n_sensors: int, latent_dim: int = 64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(n_sensors, 16, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=5, padding=2),
            nn.ReLU(),
        )
        # 对时间维做全局平均池化 -> (B, 32)
        self.pool = nn.AdaptiveAvgPool1d(1)  # 捕捉温度平均值
        self.ln   = nn.LayerNorm(32)
        self.fc   = nn.Linear(32, latent_dim)

    def forward(self, x):
        # x: (B, n_sensors, Nt)
        h = self.conv(x)              # (B, 32, Nt)
        h = self.pool(h).squeeze(-1)  # (B, 32)
        h = self.ln(h)
        b = self.fc(h)                # (B, latent_dim = p)
        return b


class TrunkNet(nn.Module):

    def __init__(self, latent_dim: int = 64, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, latent_dim)
        )

    def forward(self, T_query):          # T_query 必须是顺序向量
        # 确保是 1D tensor
        if T_query.dim() == 1:
            x = T_query.unsqueeze(-1)    # (n_T, 1)
        else:
            # 如果是 (B, n_T)，也兼容
            x = T_query.unsqueeze(-1)
        t = self.net(x)                  # (..., latent_dim)
        return t


class InverseDeepONetMonotone(nn.Module):

    def __init__(self,
                 n_sensors: int,
                 latent_dim: int = 64,
                 theta_sat: float = 0.5):
        super().__init__()
        self.branch     = BranchCNN(n_sensors, latent_dim)
        self.trunk      = TrunkNet(latent_dim)
        self.theta_sat  = theta_sat
        self.latent_dim = latent_dim

        # 由 branch 的隐向量预测每个样本的残余含水量系数 (0~1)
        self.fc_res = nn.Linear(latent_dim, 1)

        # 增加一个“陡峭度/尺度”自由度（每个样本 1 个标量）
        self.fc_scale = nn.Linear(latent_dim, 1)

    def forward(self, X_sensor, T_query):
        
        device = X_sensor.device
        T_query = T_query.to(device)

        B   = X_sensor.size(0)
        n_T = T_query.numel()

        # -------- 1. 分离 T<0 和 T>=0 --------
        mask_neg = (T_query < 0)          # (n_T,)
        mask_pos = ~mask_neg              # (n_T,)

        idx_neg = torch.nonzero(mask_neg, as_tuple=False).squeeze(-1)  # (n_T_neg,)
        idx_pos = torch.nonzero(mask_pos, as_tuple=False).squeeze(-1)  # (n_T_pos,)

        T_neg = T_query[idx_neg]          # (n_T_neg,)

        # -------- 2. branch: 传感器时序 -> b(u), 再得到 θ_res --------
        # b: (B, latent_dim)
        b = self.branch(X_sensor)         # (B, latent_dim)

        # 每个样本的残余含水量 θ_res ∈ (0, theta_sat)
        # 先映射到 (0,1)，再乘以 theta_sat
        theta_res_frac = torch.sigmoid(self.fc_res(b))   # (B, 1), 0~1
        theta_res = theta_res_frac * self.theta_sat      # (B, 1)

        # -------- 3. trunk: T<0 -> t(T) --------
        t_neg = self.trunk(T_neg)         # (n_T_neg, latent_dim)

        # -------- 4. DeepONet 组合得到一个单调的 0~1 形状函数 s(T) --------
        # raw: (B, n_T_neg)
        theta_raw_neg = torch.matmul(b, t_neg.T)   # (B, n_T_neg)

        # softplus 确保增量非负
        scale = 0.2 + 5.0 * torch.sigmoid(self.fc_scale(b))   # (B,1) > 0
        delta = F.softplus(theta_raw_neg) * scale              # (B,n_T_neg)

        # 沿 T 维累积和 -> 严格单调非减
        theta_cum = torch.cumsum(delta, dim=1)     # (B, n_T_neg)

        eps = 1e-6
        # 归一化到 [0,1]，最后一点 = 1
        s_neg = theta_cum / (theta_cum[:, -1:].clone() + eps)   # (B, n_T_neg), 0~1

        # -------- 5. 映射到 [θ_res, θ_sat] --------
        # 利用广播：theta_res: (B,1)  s_neg: (B,n_T_neg)
        theta_neg = theta_res + (self.theta_sat - theta_res) * s_neg  # (B, n_T_neg)

        # -------- 6. 组装完整 θ(T) --------
        theta = torch.zeros(B, n_T, device=device)

        if idx_neg.numel() > 0:
            theta[:, idx_neg] = theta_neg

        if idx_pos.numel() > 0:
            theta[:, idx_pos] = self.theta_sat

        return theta

4. 加载 DeepONet

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

n_case, n_sensors, Nt = X_sensor_clean.shape
latent_dim = 8
theta_sat = 0.5

deeponet_eval = InverseDeepONetMonotone(
    n_sensors=n_sensors,
    latent_dim=latent_dim,
    theta_sat=theta_sat,
).to(device)

state_dict = torch.load(
    "deeponet_sfcc_operator_train_80.pt",
    map_location=device
)

deeponet_eval.load_state_dict(state_dict)
deeponet_eval.eval()

print("Loaded Train80 SC-DION model.")

Using device: cuda
Loaded Train80 SC-DION model.


5. Robustness to missing sensor observations

5.1 Reference SFCCs

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

theta_sat = 0.5  # 之前统一的饱和体积含水量

param_cols = ["form_id", "alpha", "beta", "theta_res", "gamma", "n"]
params_df = pd.DataFrame(params_arr, columns=param_cols)
params_df["curve_id"] = curve_ids
print(params_df.head())


   form_id  alpha  beta  theta_res     gamma    n  curve_id
0      2.0    0.0   0.0   0.099743  1.430308  0.0         1
1      2.0    0.0   0.0   0.101682  0.833243  0.0         2
2      2.0    0.0   0.0   0.098324  0.501503  0.0         3
3      2.0    0.0   0.0   0.176258  0.801955  0.0         4
4      2.0    0.0   0.0   0.172534  0.616955  0.0         5


In [ ]:
def theta_power_law(T, alpha, beta, theta_sat=0.5):
    T = np.asarray(T, dtype=float)
    x = np.maximum(-T, 1e-6)
    val = alpha * x**beta
    return np.clip(val, 0.0, theta_sat)

# θ = θ_res + (θ_sat - θ_res) * exp(-((T - T0)/γ)^2)
def theta_exponential(T, theta_res, gamma, theta_sat=0.5, T0=0.0):
    T = np.asarray(T, dtype=float)
    val = theta_res + (theta_sat - theta_res) * np.exp(-((T - T0) / gamma)**2)
    return np.clip(val, 0.0, theta_sat)

# VG
def theta_vg(T, theta_res, alpha, n, theta_sat=0.5):
    T = np.asarray(T, dtype=float)
    x = np.maximum(-T, 1e-6)   # 防止 0
    m = 1.0 - 1.0 / n
    val = theta_res + (theta_sat - theta_res) * (1.0 + (alpha * x)**n)**(-m)
    return np.clip(val, 0.0, theta_sat)

# 统一接口：
def theta_w(T, param_row, theta_sat=0.5):
    form = int(param_row["form_id"])  # 1, 2, or 3

    if form == 1:
        return theta_power_law(
            T,
            param_row["alpha"],
            param_row["beta"],
            theta_sat=theta_sat,
        )

    elif form == 2:
        return theta_exponential(
            T,
            param_row["theta_res"],
            param_row["gamma"],
            theta_sat=theta_sat,
            T0 = 0.0
        )

    elif form == 3:
        return theta_vg(
            T,
            param_row["theta_res"],
            param_row["alpha"],
            param_row["n"],
            theta_sat=theta_sat
        )
        
    else:
        raise ValueError(f"未知 form_id={form}，期望是 1/2/3")


In [20]:
# 评估用温度网格（和训练范围一致即可）
T_eval_np = np.linspace(-15.0, 0.0, 50).astype("float32")
n_T_eval = T_eval_np.shape[0]
n_case = params_arr.shape[0]

theta_true_np = np.zeros((n_case, n_T_eval), dtype=np.float32)

for i in range(n_case):
    row = params_df.iloc[i]
    theta_true_np[i] = theta_w(T_eval_np, row, theta_sat=theta_sat)

print("theta_true_np shape:", theta_true_np.shape)   # (n_case, 50)


theta_true_np shape: (220, 50)


5.2 SFCC inversion with missing sensors

In [21]:
import numpy as np
import pandas as pd
import torch

from itertools import combinations


def fill_missing_sensors(
    temperature_data,
    sensor_depths,
    missing_indices
):
    """
    使用其余有效传感器沿深度方向对缺失传感器进行线性插值。

    Parameters
    ----------
    temperature_data : ndarray
        Shape: (n_case, Nt, n_sensor)

    sensor_depths : ndarray
        Shape: (n_sensor,)

    missing_indices : tuple or list
        缺失传感器的索引。

    Returns
    -------
    filled_data : ndarray
        Shape: (n_case, Nt, n_sensor)
    """

    filled_data = np.asarray(
        temperature_data,
        dtype=np.float32
    ).copy()

    sensor_depths = np.asarray(
        sensor_depths,
        dtype=np.float32
    )

    missing_indices = tuple(
        sorted(missing_indices)
    )

    if len(missing_indices) == 0:
        return filled_data

    n_sensor = filled_data.shape[2]

    available_indices = [
        i for i in range(n_sensor)
        if i not in missing_indices
    ]

    for missing_id in missing_indices:

        # 缺失位置左侧的有效传感器
        left_candidates = [
            i for i in available_indices
            if i < missing_id
        ]

        # 缺失位置右侧的有效传感器
        right_candidates = [
            i for i in available_indices
            if i > missing_id
        ]

        if left_candidates and right_candidates:

            left_id = max(left_candidates)
            right_id = min(right_candidates)

            z_left = sensor_depths[left_id]
            z_right = sensor_depths[right_id]
            z_missing = sensor_depths[missing_id]

            weight = (
                (z_missing - z_left)
                / (z_right - z_left)
            )

            filled_data[:, :, missing_id] = (
                (1.0 - weight)
                * temperature_data[:, :, left_id]
                + weight
                * temperature_data[:, :, right_id]
            )

        elif right_candidates:

            # 最浅传感器缺失：使用最近的有效传感器
            right_id = min(right_candidates)

            filled_data[:, :, missing_id] = (
                temperature_data[:, :, right_id]
            )

        elif left_candidates:

            # 最深传感器缺失：使用最近的有效传感器
            left_id = max(left_candidates)

            filled_data[:, :, missing_id] = (
                temperature_data[:, :, left_id]
            )

        else:

            raise ValueError(
                "No available sensors for interpolation."
            )

    return filled_data

In [22]:
# =========================================================
# Missing-sensor robustness test
# =========================================================

T_eval = torch.from_numpy(
    T_eval_np
).float().to(device)

sensor_z_np = np.asarray(
    sensor_z,
    dtype=np.float32
)

# 测试缺失0、1、2、3个传感器
missing_counts = [0, 1, 2, 3, 4, 5]

# reference SFCC的global R2分母固定不变
ss_tot_global = np.sum(
    (
        theta_true_np
        - np.mean(theta_true_np)
    )**2
)

missing_sensor_results = {}

deeponet_eval.eval()

for n_missing in missing_counts:

    if n_missing == 0:

        configurations = [tuple()]

    else:

        configurations = list(
            combinations(
                range(n_sensors),
                n_missing
            )
        )

    rmse_values = []
    r2_values = []

    print(
        f"\nMissing sensors: {n_missing}; "
        f"configurations: {len(configurations)}"
    )

    for missing_configuration in configurations:

        # 1. 对缺失传感器进行深度插值
        all_T_sensor_filled = fill_missing_sensors(
            temperature_data=all_T_sensor_clean,
            sensor_depths=sensor_z_np,
            missing_indices=missing_configuration
        )

        # 2. 转换为网络输入格式
        # (n_case, Nt, n_sensor)
        # -> (n_case, n_sensor, Nt)
        X_sensor_filled = torch.from_numpy(
            all_T_sensor_filled
        ).permute(0, 2, 1).float().to(device)

        # 3. 使用原Train80模型直接推理
        with torch.no_grad():

            theta_pred = deeponet_eval(
                X_sensor_filled,
                T_eval
            )

        theta_pred_np = (
            theta_pred
            .cpu()
            .numpy()
        )

        # 4. Global SFCC metrics
        err_global = (
            theta_pred_np
            - theta_true_np
        )

        global_rmse = np.sqrt(
            np.mean(err_global**2)
        )

        ss_res_global = np.sum(
            err_global**2
        )

        global_r2 = (
            1.0
            - ss_res_global / ss_tot_global
        )

        rmse_values.append(global_rmse)
        r2_values.append(global_r2)

    missing_sensor_results[n_missing] = {
        "configurations": configurations,
        "rmse": np.asarray(
            rmse_values,
            dtype=np.float64
        ),
        "r2": np.asarray(
            r2_values,
            dtype=np.float64
        )
    }

print("\nMissing-sensor test completed.")


Missing sensors: 0; configurations: 1

Missing sensors: 1; configurations: 11

Missing sensors: 2; configurations: 55

Missing sensors: 3; configurations: 165

Missing sensors: 4; configurations: 330

Missing sensors: 5; configurations: 462

Missing-sensor test completed.


In [23]:
summary_rows = []

for n_missing in missing_counts:

    rmse_values = (
        missing_sensor_results[n_missing]["rmse"]
    )

    r2_values = (
        missing_sensor_results[n_missing]["r2"]
    )

    n_configuration = len(rmse_values)

    if n_configuration > 1:

        rmse_std = rmse_values.std(ddof=1)
        r2_std = r2_values.std(ddof=1)

    else:

        rmse_std = 0.0
        r2_std = 0.0

    summary_rows.append({
        "Available sensors": n_sensors - n_missing,
        "Missing sensors": n_missing,
        "Number of configurations": n_configuration,
        "Global RMSE mean": rmse_values.mean(),
        "Global RMSE std": rmse_std,
        "Global R2 mean": r2_values.mean(),
        "Global R2 std": r2_std,
        "Global RMSE max": rmse_values.max(),
        "Global R2 min": r2_values.min()
    })

missing_sensor_summary = pd.DataFrame(
    summary_rows
)

display(missing_sensor_summary)

,Available sensors,Missing sensors,Number of configurations,Global RMSE mean,Global RMSE std,Global R2 mean,Global R2 std,Global RMSE max,Global R2 min
0,11,0,1,0.032020,0.000000,0.908524,0.000000,0.032020,0.908524
1,10,1,11,0.038510,0.020300,0.834258,0.239223,0.099706,0.113001
2,9,2,55,0.046011,0.027891,0.742965,0.353996,0.136999,-0.674592
3,8,3,165,0.054663,0.033822,0.631945,0.457168,0.143619,-0.840365
4,7,4,330,0.064598,0.038146,0.498244,0.545090,0.143715,-0.842826
5,6,5,462,0.075911,0.040624,0.338919,0.611198,0.143913,-0.847907


In [24]:
for n_missing in [1, 2, 3, 4, 5]:

    result = missing_sensor_results[n_missing]

    rmse_values = result["rmse"]
    configurations = result["configurations"]

    worst_id = int(
        np.argmax(rmse_values)
    )

    worst_configuration = (
        configurations[worst_id]
    )

    worst_depths = sensor_z_np[
        list(worst_configuration)
    ]

    print(
        f"Missing {n_missing} sensor(s): "
        f"worst depths = {worst_depths}, "
        f"RMSE = {rmse_values[worst_id]:.6f}, "
        f"R2 = {result['r2'][worst_id]:.6f}"
    )

Missing 1 sensor(s): worst depths = [0.55], RMSE = 0.099706, R2 = 0.113001
Missing 2 sensor(s): worst depths = [0.5  0.55], RMSE = 0.136999, R2 = -0.674592
Missing 3 sensor(s): worst depths = [0.45 0.5  0.55], RMSE = 0.143619, R2 = -0.840365
Missing 4 sensor(s): worst depths = [0.3  0.45 0.5  0.55], RMSE = 0.143715, R2 = -0.842826
Missing 5 sensor(s): worst depths = [0.3  0.35 0.45 0.5  0.55], RMSE = 0.143913, R2 = -0.847907
